# P03 最初のサーバ ― 公式 SDK で 1 歩ずつ書く

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

社内規程 13 本を探すサーバを mcp 2.3.0 の `MCPServer` で書き、`Client` でつないで確かめます。ステップ 1 は道具 1 つ、ステップ 2 で実務の形 (構造化出力・注釈・ToolError・資料・定型文) に育てます。

- 上から順に実行します。0 のセルは準備です (Colab でも手元の Jupyter でも同じ)。
- このノートブックは Claude を呼びません。API キーが無くても全部動きます。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る (Colab のときだけ取得とインストールをする)
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/mcp-skills-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/mcp-skills-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{KIT}/requirements.txt"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根 (common/ がある所) を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/host.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. このノートブックは Claude を呼びません (API キー不要)。ラボのモジュールを読めるようにする
from common.nb import safe
from common.kit import use_labs, utf8_console
use_labs(); utf8_console()

## 1. ステップ 1: 型ヒントと docstring から道具の形ができる
`step1_server.py` は関数 1 つに `@mcp.tool()` を付けただけです。関数名・docstring・型ヒント・既定値から、SDK が tools/list の中身を作ります。

In [ ]:
from run_first_server import amain as first
with safe():
    r = await first(["--step", "1"])

## 2. 生成された tools/list をそのまま見る
モデルが受け取るのはこの JSON です。docstring の `Args:` が description に入っただけで、引数の説明は空のままなことを確かめます。

In [ ]:
with safe():
    r = await first(["--step", "1", "--json"])

## 3. ステップ 2: 構造化出力・注釈・ToolError・資料・定型文
戻り値を pydantic のモデルにすると `structured_content` が付きます。無い番号は `ToolError` で断り、直し方 (ある番号) を添えます。

In [ ]:
with safe():
    r = await first(["--step", "2"])

## 4. ToolError と想定外の例外の違い
同じ「無い番号」でも、`ToolError` で断ると文が届き、素朴に `KeyError` を起こすと「Error executing tool …」しか届きません。どれも例外ではなく `is_error=True` の結果です。

In [ ]:
with safe():
    r = await first(["--errors"])

## 5. 同じサーバを子プロセス (stdio) で立ててつなぐ
Claude Desktop・Claude Code と同じつなぎ方です。サーバのコードは変えず、`Client` に渡すものだけが変わります。

In [ ]:
with safe():
    r = await first(["--step", "2", "--stdio"])